In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from tensorflow.examples.tutorials.mnist import input_data
mnist = input_data.read_data_sets('./mnist/data/', one_hot=True)

Successfully downloaded train-images-idx3-ubyte.gz 9912422 bytes.
Extracting ./mnist/data/train-images-idx3-ubyte.gz
Successfully downloaded train-labels-idx1-ubyte.gz 28881 bytes.
Extracting ./mnist/data/train-labels-idx1-ubyte.gz
Successfully downloaded t10k-images-idx3-ubyte.gz 1648877 bytes.
Extracting ./mnist/data/t10k-images-idx3-ubyte.gz
Successfully downloaded t10k-labels-idx1-ubyte.gz 4542 bytes.
Extracting ./mnist/data/t10k-labels-idx1-ubyte.gz


In [3]:
total_epoch = 100
batch_size = 100
learning_rate = 0.0002

n_input = 28 * 28
n_noise = 128
n_hidden = 256

In [4]:
X = tf.placeholder(tf.float32, [None, n_input])
Z = tf.placeholder(tf.float32, [None, n_noise])

G_W1 = tf.Variable(tf.random_normal([n_noise, n_hidden], stddev=0.01))
G_b1 = tf.Variable(tf.zeros([n_hidden]))
G_W2 = tf.Variable(tf.random_normal([n_hidden, n_input], stddev=0.01))
G_b2 = tf.Variable(tf.zeros([n_input]))

D_W1 = tf.Variable(tf.random_normal([n_input, n_hidden], stddev=0.01))
D_b1 = tf.Variable(tf.zeros([n_hidden]))
D_W2 = tf.Variable(tf.random_normal([n_hidden, 1], stddev=0.01))
D_b2 = tf.Variable(tf.zeros([1]))

In [5]:
def generator(noise_z):
    hidden = tf.nn.relu(tf.matmul(noise_z, G_W1) + G_b1)
    output = tf.nn.sigmoid(tf.matmul(hidden, G_W2) + G_b2)
    
    return output

def discriminator(inputs):
    hidden = tf.nn.relu(tf.matmul(inputs, D_W1) + D_b1)
    output = tf.nn.sigmoid(tf.matmul(hidden, D_W2) + D_b2)
    
    return output

def get_noise(batch_size, n_noise):
    return np.random.normal(size=(batch_size, n_noise))

In [6]:
G = generator(Z)
D_gene = discriminator(G)
D_real = discriminator(X)

In [7]:
loss_D = tf.reduce_mean(tf.log(D_real) + tf.log(1 - D_gene))
loss_G = tf.reduce_mean(tf.log(D_gene))

D_var_list = [D_W1, D_b1, D_W2, D_b2]
G_var_list = [G_W1, G_b1, G_W2, G_b2]

train_D = tf.train.AdamOptimizer(learning_rate).minimize(-loss_D, var_list=D_var_list)
train_G = tf.train.AdamOptimizer(learning_rate).minimize(-loss_G, var_list=G_var_list)

In [8]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())

total_batch = int(mnist.train.num_examples / batch_size)
loss_val_D, loss_val_G = 0, 0

for epoch in range(total_epoch):
    for i in range(total_batch):
        batch_xs, batch_ys = mnist.train.next_batch(batch_size)
        noise = get_noise(batch_size, n_noise)
        
        _, loss_val_D = sess.run([train_D, loss_D], feed_dict={X: batch_xs, Z: noise})
        _, loss_val_G = sess.run([train_G, loss_G], feed_dict={Z: noise})
    
    print('Epoch:', '%04d' % epoch, 'D loss: {:.4}'.format(loss_val_D), 'G loss: {:.4}'.format(loss_val_G))

    if epoch == 0 or (epoch + 1) % 10 == 0:
        sample_size = 10
        noise = get_noise(sample_size, n_noise)
        samples = sess.run(G, feed_dict={Z: noise})

        fig, ax = plt.subplots(1, sample_size, figsize=(sample_size, 1))

        for i in range(sample_size):
            ax[i].set_axis_off()
            ax[i].imshow(np.reshape(samples[i], (28, 28)))

        plt.savefig('./samples/{}.png'.format(str(epoch).zfill(3)), bbox_inches='tight')
        plt.close(fig)

print('Training Completed !')

Epoch: 0000 D loss: -0.5415 G loss: -1.989
Epoch: 0001 D loss: -0.3072 G loss: -2.46
Epoch: 0002 D loss: -0.06106 G loss: -3.65
Epoch: 0003 D loss: -0.411 G loss: -1.734
Epoch: 0004 D loss: -0.4694 G loss: -1.815
Epoch: 0005 D loss: -0.3417 G loss: -2.217
Epoch: 0006 D loss: -0.2683 G loss: -2.563
Epoch: 0007 D loss: -0.2275 G loss: -3.037
Epoch: 0008 D loss: -0.2415 G loss: -2.67
Epoch: 0009 D loss: -0.3247 G loss: -2.986
Epoch: 0010 D loss: -0.2698 G loss: -2.678
Epoch: 0011 D loss: -0.4698 G loss: -2.208
Epoch: 0012 D loss: -0.22 G loss: -2.715
Epoch: 0013 D loss: -0.3448 G loss: -2.657
Epoch: 0014 D loss: -0.3668 G loss: -2.398
Epoch: 0015 D loss: -0.3985 G loss: -2.233
Epoch: 0016 D loss: -0.4312 G loss: -2.272
Epoch: 0017 D loss: -0.3885 G loss: -2.649
Epoch: 0018 D loss: -0.2432 G loss: -2.937
Epoch: 0019 D loss: -0.2529 G loss: -2.88
Epoch: 0020 D loss: -0.3531 G loss: -2.942
Epoch: 0021 D loss: -0.2882 G loss: -2.903
Epoch: 0022 D loss: -0.3086 G loss: -2.775
Epoch: 0023 D los